# 01 — Basic Agent

This notebook demonstrates the simplest possible orqest agent: a single agent that takes a user message, runs it through an LLM, and returns structured output.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4o   # or anthropic:claude-sonnet-4-20250514, google:gemini-2.0-flash, etc.
  ```

## 1. Load config

orqest never loads environment variables at import time — you call `load_config()` explicitly.

In [1]:
from orqest import load_config

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

Model:    openai:gpt-5.2
API key:  sk-proj-...


## 2. Define the agent

Every orqest agent needs three things:
1. **A state type** — the input (here we use the built-in `GlobalState`)
2. **An output type** — a Pydantic model defining the structured response
3. **`_run_implementation()`** — the async method where your logic lives

In [2]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


class SummaryOutput(BaseModel):
    """The agent's structured output."""
    summary: str = Field(description="A concise summary of the user's message")
    key_points: list[str] = Field(description="Bullet points of the main ideas")


class SummaryAgent(BaseAgent[GlobalState, SummaryOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="summary_agent",
            system_prompt=(
                "You are a summarization assistant. Given a user's message, "
                "produce a concise summary and a list of key points."
            ),
            output_type=SummaryOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SummaryOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output

## 3. Run the agent

Create a `GlobalState`, add a user message, and call `agent.run()`.

In [3]:
agent = SummaryAgent(model=config.llm_model, api_key=config.llm_api_key)

state = GlobalState()
state.add_message(
    "user",
    "Quantum computing uses qubits instead of classical bits. Unlike bits that are "
    "either 0 or 1, qubits can exist in superposition — representing both states "
    "simultaneously. This allows quantum computers to explore many solutions in "
    "parallel. Combined with entanglement, where qubits become correlated regardless "
    "of distance, quantum computers can solve certain problems exponentially faster "
    "than classical machines. Key applications include cryptography, drug discovery, "
    "optimization problems, and materials science."
)

output = await agent.run(state)

print("Summary:")
print(output.summary)
print("\nKey points:")
for point in output.key_points:
    print(f"  - {point}")

Summary:
The message explains how quantum computing differs from classical computing by using qubits that can be in superposition and become entangled, enabling parallel exploration of solutions and speedups for specific problems.

Key points:
  - Classical bits are 0 or 1; qubits can be in superposition of both states.
  - Superposition allows quantum computers to explore many possibilities in parallel.
  - Entanglement creates strong correlations between qubits independent of distance.
  - These properties can yield exponential speedups for certain problem classes.
  - Potential applications include cryptography, drug discovery, optimization, and materials science.


## What's happening under the hood

1. `load_config()` reads your `.env` and validates that `LLM_API_KEY` is set
2. `SummaryAgent.__init__()` calls `resolve_model()` which maps `"openai:gpt-4o"` to the correct pydantic-ai `Model` + `Provider`
3. On first access to `agent.agent`, a pydantic-ai `Agent` is lazily constructed with the system prompt, output type, and model
4. `agent.run(state)` calls `_run_implementation()` which extracts the user message and calls `self.call_model()`
5. `call_model()` passes `state.message_history` to pydantic-ai and stores the updated history back on state
6. pydantic-ai validates the LLM response against `SummaryOutput` and returns typed, structured data

## 4. Multi-turn conversation

Because `_run_implementation` uses `self.call_model()`, conversation history is automatically managed. Each call stores the full message history on `state.message_history`, so the next call picks up where the previous one left off.

Let's send a follow-up message using the **same state** — the agent will have context from the first turn.

In [4]:
# Check the history after the first turn
print(f"Messages in history after turn 1: {len(state.message_history)}")
for msg in state.message_history:
    print(f"  {msg.kind}: {msg.parts[0]}")

Messages in history after turn 1: 3
  request: SystemPromptPart(content="You are a summarization assistant. Given a user's message, produce a concise summary and a list of key points.", timestamp=datetime.datetime(2026, 5, 15, 9, 27, 6, 33891, tzinfo=datetime.timezone.utc))
  response: ToolCallPart(tool_name='final_result', args='{"summary":"The message explains how quantum computing differs from classical computing by using qubits that can be in superposition and become entangled, enabling parallel exploration of solutions and speedups for specific problems.","key_points":["Classical bits are 0 or 1; qubits can be in superposition of both states.","Superposition allows quantum computers to explore many possibilities in parallel.","Entanglement creates strong correlations between qubits independent of distance.","These properties can yield exponential speedups for certain problem classes.","Potential applications include cryptography, drug discovery, optimization, and materials science

In [5]:
# Second turn — the agent has context from the first turn
state.add_message(
    "user",
    "Now focus only on the cryptography applications you mentioned. "
    "Summarize how quantum computing specifically impacts cryptography."
)

output2 = await agent.run(state)

print("Follow-up summary:")
print(output2.summary)
print("\nKey points:")
for point in output2.key_points:
    print(f"  - {point}")

print(f"\nMessages in history after turn 2: {len(state.message_history)}")

Follow-up summary:
Quantum computing affects cryptography by threatening widely used public‑key schemes and enabling new quantum-safe methods for key exchange and security.

Key points:
  - Shor’s algorithm could break RSA and elliptic-curve cryptography by efficiently factoring and solving discrete logs.
  - Grover’s algorithm speeds up brute-force search, effectively reducing symmetric-key security (e.g., requiring longer AES keys).
  - This drives adoption of post-quantum cryptography (PQC): classical algorithms designed to resist quantum attacks.
  - Quantum cryptography/QKD can provide key distribution with security based on physics rather than computational hardness.
  - A “harvest now, decrypt later” risk exists: encrypted data captured today may be decrypted in the future once large quantum computers exist.

Messages in history after turn 2: 6


## Multi-turn: how it works

- `self.call_model(prompt, state)` passes `state.message_history` into pydantic-ai's `Agent.run(message_history=...)`
- After the run, it stores the updated history back: `state.message_history = result.all_messages()`
- History accumulates across turns — the LLM sees the full conversation
- The `keep_recent_messages` history processor (configured via `truncated_history`) automatically truncates old messages when the history gets too long

**Stateless vs. stateful**: If you don't need history (e.g., agent-as-tool pattern), use `self.agent.run()` directly instead of `self.call_model()`. See example 02.